In [1]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool
from com.example.agentic.agent.LLMManager import LLMManager

# groq, openai
llm = LLMManager.create_llm('ollama')

llm.call('Hello')

'Hello. Is there something I can help you with or would you like to chat?'

In [2]:
# Setting up common configurer for tool
from typing import Literal, cast

_tool_config = dict(
    llm=dict(
        provider="ollama", # or google, openai, anthropic, llama2, ...
        config=dict(
            model="llama3.2:1b-instruct-q8_0",
            # temperature=0.5,
            # top_p=1,
            # stream=true,
        ),
    ),
    embedder=dict(
        provider="ollama", # or openai, ollama, ...
        config=dict(
            model_name="all-minilm",
            task_type="RETRIEVAL_DOCUMENT",
            # title="Embeddings",
        ),
    ),
)

In [3]:
# Setting up common configurer for rag tools
_rag_tool_config = {
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "vectordb": {
        "provider": "chromadb",
        "config": {
            "persist_directory":"agentic-ai/chromadb",
            "allow_reset":"true",
            "is_persistent":"true",
            "settings":{
                "persist_directory":"agentic-ai/chromadb", 
                "allow_reset":True, 
                "is_persistent":True
            }
        }
    },
}
   

In [ ]:
from crewai_tools import CodeDocsSearchTool

@tool
def codedoc_search_tool(doc_path: str):
    """
    read api documentation and return content.
    """
    # by providing its URL:
    codedoc_search_tool = CodeDocsSearchTool(
        docs_url='https://docs.oracle.com/javase/8/docs/api/java/net/URL.html',
        config=_tool_config
    )

    #
    return codedoc_search_tool.run()

In [5]:
import os
from crewai_tools import GithubSearchTool

# https://github.com/
_repo = 'brijeshdhaker/docker-hadoop-cluster'
_git_hub_token = os.getenv("GIT_HUB_TOKEN")

github_search_tool = GithubSearchTool(
    config=_tool_config,
    gh_token=_git_hub_token,
)

#github_search_tool.add(repo=_repo, content_types=['code', 'issue'])
github_search_tool.run(search_query="hadoop", 
    github_repo="brijeshdhaker/docker-hadoop-cluster", 
    content_types=['code', 'issue']
)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} in upsert.

In [ ]:
from crewai_tools import DOCXSearchTool
# Instantiate Web Search Tool

#@tool
def docx_search_tool(query: str, doc_path :str):
    """
    read workd document and return document content.
    """
	# Initialize the tool for semantic searches within a specific GitHub repository
    _doc_path = '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx'
    if doc_path :
        _doc_path = doc_path

    # Initialize the tool to search within any DOCX file's content
    docx_tool = DOCXSearchTool(docx=_doc_path, config=_rag_tool_config)
    #
    return docx_tool.run(query)


#
docx_results = docx_search_tool(query="Cloudera", doc_path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx")
docx_results

In [ ]:
from crewai.tools import tool
from crewai_tools import WebsiteSearchTool

# Instantiate Web Search Tool
@tool
def web_search_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )
    #
    return search_tool.run(query)


In [ ]:
from crewai_tools import ScrapeWebsiteTool


# Instantiate Web Search Tool
@tool
def scrap_website_tool(query: str):
    """
    scrap the content of the specified website.
    """
    # Initialize the tool with the website URL, 
    # so the agent can only scrap the content of the specified website
    scrap_website_tool = ScrapeWebsiteTool(
        website_url='https://example.com',
        config=_tool_config
    )
    
    # Extract the text from the site
    #text = scrap_website_tool.run()
    #print(text)

    #
    return scrap_website_tool.run(query)

In [ ]:
# 1. Initialize the tool
from crewai_tools import PDFSearchTool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
@tool
def pdf_search_tool(query : str):
        """_summary
            Searches the pdf documents and returns results.
        """
        pdf_tool = PDFSearchTool(
            pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
            config={
                "embedding_model": {
                    "provider": "openai",
                    "config": {
                        "model": "nomic-embed-text",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "embedder": {
                    "provider": "openai",
                    "config": {
                        "model": "all-minilm",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        "persist_directory":"agentic-ai/chromadb",
                        "allow_reset":"true",
                        "is_persistent":"true",
                    }
                },
            }
        )

        return pdf_tool.run(query)

#
results = pdf_search_tool.run("Cloudera")
results

In [ ]:
from crewai_tools import TavilyExtractorTool
# Initialize with custom configuration
extractor_tool = TavilyExtractorTool(
    extract_depth='advanced',  # More comprehensive extraction
    include_images=True,       # Include image results
    timeout=90                 # Custom timeout
)




#### LOAD Documents

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(data)
documents

##### Embaddings

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

#### Formatters

In [6]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key images or design about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Source image with title and url for each findings",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [9]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [10]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

##### Design Crew

In [11]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from com.example.agentic.tools.DesignSearchTool import DesignSearchTool, DesignInput

@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True,
            tools=[DesignSearchTool()],
            #knowledge_sources=[text_kw_source],
            embedder={
                "provider": "openai",
                "config": {
                    "model": "nomic-embed-text",
                    "api_key":"",
                    "platform_url":"http://localhost:11434/v1"
                },
            },
            llm=llm
        )
    
    @agent
    def image_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['image_researcher'],
            verbose=True,
            tools=[],
            allow_delegation=False,
            llm=llm
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True,
            llm=llm
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True,
            llm=llm
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'],
            output_json=ResearchOutput
        )
    
    @task
    def find_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['find_images_task'],
            output_json=ResearchImageOutput
        )
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )
		
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [12]:
from datetime import datetime

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': datetime.now().strftime('%Y-%m-%d')
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

In [13]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ca6e7f6b-6fe4-4370-961d-c2adf68705ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: 7c7ef331-1976-4adf-a2fd-34a3a62708a4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find any interesting and relevant    │
│  information about Microservice Design.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='Microservices can be designed to be      │
│  scalable, maintainable, and easy to test.', relevance='This finding is relevant because it highlights the      │
│  benefits of using microservices in software design, such as separation of duties and improved scalability.',   │
│  sources=[{'additionalProperties': 'default', 'type': 'object', 'properties': '', 'required': 'All'}]),         │
│  ResearchPoint(topic='Microservice Interoperability', findings='Microservices can be designed to interact with  │
│  each other seamlessly through APIs and message queues.', relevance='This finding is relevant because it        │
│  emphasizes the importance of designing microservices for interoperability and integration, which is critical   │
│  for large-scale deployments.', sources=[{'additionalProperties': 'default', 'type': 'object', 'properties':    │
│  '', 'required': 'All'}]), ResearchPoint(topic='Microservice Consistency', findings="It's crucial to ensure     │
│  consistency across microservices to maintain a stable and predictable behavior.", relevance='This finding is   │
│  relevant because it highlights the importance of ensuring that microservices interact with each other in a     │
│  consistent manner, which is critical for complex systems.', sources=[{'additionalProperties': 'default',       │
│  'type': 'object', 'properties': '', 'required': 'All'}]), ResearchPoint(topic='Microservice Testing',          │
│  findings='Automated testing is essential to ensure the quality and reliability of microservices.',             │
│  relevance='This finding is relevant because it stresses the importance of using automated testing techniques   │
│  to ensure the correctness and coverage of microservices in development workflows.',                            │
│  sources=[{'additionalProperties': 'default', 'type': 'object', 'properties': '', 'required': 'All'}]),         │
│  ResearchPoint(topic='Microservice Security', findings='Microservices can be designed with security in mind     │
│  through the use of authentication, authorization, and encryption.', relevance='This finding is relevant        │
│  because it highlights the importance of designing microservices with security considerations to protect data   │
│  and ensure the integrity of the system.', sources=[{'additionalProperties': 'default', 'type': 'object',       │
│  'properties': '', 'required': 'All'}]), ResearchPoint(topic='Microservice Deployment', findings='Deployment    │
│  should be done on a scalable infrastructure to ensure that the system can handle changes and updates           │
│  efficiently.', relevance='This finding is relevant because it emphasizes the importance of designing           │
│  microservices for deployment, which is critical for large-scale deployments like cloud native applications.',  │
│  sources=[{'additionalProperties': 'default', 'type': 'object', 'properties': '', 'required': 'All'}]),         │
│  ResearchPoint(topic='Microservice Monitoring', findings='Monitoring is essential to detect issues and make     │
│  data-driven decisions about system performance.', relevance='This finding is relevant because it highlights    │
│  the importance of monitoring microservices in real-time to ensure that the system operates smoothly and        │
│  efficiently.', sources=[{'additionalProperties': 'defa

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: 1 validation error for ExecutiveReport                                          │
│    Invalid JSON: lone leading surrogate in hex escape at line 49 column 32 [type=json_invalid, input_value='{   │
│  "report_title": "Micro...s": [],\n"sources": []}', input_type=str]                                             │
│      For further information visit https://errors.pydantic.dev/2.11/v/json_invalid                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: reporting_task                                                                                           │
│  Agent: Microservice Design Reporting Analyst                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: ca6e7f6b-6fe4-4370-961d-c2adf68705ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ValidationError: 1 validation error for ExecutiveReport
  Invalid JSON: lone leading surrogate in hex escape at line 49 column 32 [type=json_invalid, input_value='{ "report_title": "Micro...s": [],\n"sources": []}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/json_invalid

####
####